In [13]:
import numpy as np
import csv
import time
import os
import itertools
import pandas as pd
import matplotlib.pyplot as plt
import skimage
import glob
from matplotlib.lines import Line2D
from tqdm import tqdm
import pickle
import argparse
from scipy.spatial import cKDTree
from pathlib import Path
from math import comb
from joblib import Parallel, delayed
import multiprocessing

In [20]:
BASE_DIR = '/home/aplante/aplante_diskmnt/mCRC/xenium_5k/Evan/morphcia/'
gene_start_index = 13
dist_thresh = 30 

In [3]:
sample_organs = pd.read_csv('sample_organs.txt', sep='\t')

In [4]:
filenames = glob.glob('morphcia/seurat_to_python/*.csv')
lung_ids = []
colon_ids = []
liver_ids = []
breast_ids = []
section_ids = []
for filename in filenames:
    section_id = filename.split('/')[-1].split('.csv')[0]
    organ = sample_organs[sample_organs['Section ID'] == section_id]['Site']
    if len(organ) != 1:
        raise Exception(f"Unexpected number of matches for {section_id}: {len(organ)}")
    organ = organ.values[0]
    
    if organ == 'Colon':
        colon_ids.append(section_id)
    elif organ == 'Lung':
        lung_ids.append(section_id)
    elif organ == 'Liver':
        liver_ids.append(section_id)
    elif organ == 'Breast':
        breast_ids.append(section_id)
    else:
        print(f"Unknown organ '{organ}' for section_id {section_id}")

    section_ids.append(section_id)
# Optionally, print the collected IDs
print("Colon IDs:", colon_ids)
print("Lung IDs:", lung_ids)
print("Liver IDs:", liver_ids)
print("Breast IDs:", breast_ids)

cross_organ_matches = [
    ["S16-44514-E7U2", "S16-44514-B3U1"],
    ["S15-1909-A5U1", "S15-1909-D2U1"],
    ["S15-32775-D4U2", "S15-32775-A1U2"],
    ["S16-38794-A4U2", "S16-38794-E2U1"],
    ["S13-31378-E7U1", "S13-31378-B1U1"],
    ["HT472C1-S1H3FP1U1", "HT472C1-M1FP1U1"]
]


Colon IDs: ['S16-44514-E7U2', 'S15-1909-A5U1', 'S15-32775-D4U2', 'S16-38794-A4U2', 'S13-31378-E7U1', 'HT472C1-S1H3FP1U1', 'S13-18424-B5U1']
Lung IDs: ['S15-23849-A2U1', 'S15-21169-A1U1', 'S17-16442-AFR1U1', 'CM478C2-A3-Tp1Fp1U1', 'CM478C1-D4-Tp1Fp1U1', 'CM354C1-A2-Tp1Fp1U1']
Liver IDs: ['S16-44514-B3U1', 'S15-1909-D2U1', 'S16-38794-E2U1', 'S15-32775-A1U2', 'HT413C1-Th1K2A1U2', 'S13-31378-B1U1', 'HT472C1-M1FP1U1']
Breast IDs: ['S20-46186-A16U1']


In [5]:
donor_cell_type = 'Proliferative tumor'
receiver_cell_type = 'mCAF'

receiver_gene_names = [
    "BMP8A",
    "ITGA4",
    "NOTCH3",
    "NES",
    "ITGA1",
    "EDNRA",
    "NOTCH1",
    "MYH11",
    "LOXL2",
    "GREM1",
    "ASPN",
    "POSTN",
    "CCN1",
    "ANGPTL2",
    "ADAM12",
    "CCDC80",
    "PDGFRA",
    "C7"
]


In [6]:
# output_dir = Path('morphcia/seurat_to_python/feathers')
# output_dir.mkdir(parents=True, exist_ok=True)

# for sec_id in tqdm.tqdm(section_ids, desc="Converting CSVs to Feather"):
#     csv_path = Path(BASE_DIR) / f"seurat_to_python/{sec_id}.csv"
#     feather_path = output_dir / f"{sec_id}.feather"
#     if os.path.exists(feather_path):
#         print(sec_id, 'exists. skipping...')
#         continue

#     df = pd.read_csv(csv_path, low_memory=False)
    
    
#     df.to_feather(feather_path)

#     del df


####################################################################
# all_data = pd.DataFrame()

# for section_id in tqdm.tqdm(section_ids, desc="Building Master DF"):
#     feather_path = f"morphcia/seurat_to_python/feathers/{section_id}.feather"
#     df_temp = pd.read_feather(feather_path)
    
#     # 1) Identify numeric columns (float/int)
#     numeric_cols = df_temp.select_dtypes(include=["float", "int"]).columns
    
#     # 2) Convert them to float32 (halves memory vs. float64)
#     df_temp[numeric_cols] = df_temp[numeric_cols].astype(np.float32)
    
#     # 3) Make these columns sparse with fill_value=0
#     for col in numeric_cols:
#         df_temp[col] = pd.arrays.SparseArray(df_temp[col], fill_value=0)
    
#     # 4) Concatenate into the main DataFrame
#     all_data = pd.concat([all_data, df_temp], ignore_index=True)
    
#     del df_temp  # free chunk memory

# print("Final DataFrame shape:", all_data.shape)
# print("Sparse info:")
# print(all_data.info(memory_usage="deep"))


# # Identify sparse columns using pd.api.types.is_sparse
# sparse_cols = [col for col in all_data.columns if pd.api.types.is_sparse(all_data[col])]

# # Convert each sparse column to dense
# for col in sparse_cols:
#     all_data[col] = all_data[col].sparse.to_dense()


# # Now try saving
# all_data.to_feather("morphcia/seurat_to_python/all_data.feather")

In [22]:
for reciever_gene in receiver_gene_names[::-1]:
    donor_values_pos = {}
    donor_values_neg = {}
    for section_id in tqdm(section_ids):
        # print(section_id)
        # seurat_table = all_data.loc[all_data["orig.ident"] == section_id]
        seurat_table = pd.read_feather('morphcia/seurat_to_python/feathers/'+section_id+'.feather')

        receiver_cells = seurat_table[seurat_table["all_cell_type"]==receiver_cell_type]
        donor_cells = seurat_table[seurat_table["all_cell_type"]==donor_cell_type]
        gene_names = donor_cells.columns[gene_start_index:].values

        if receiver_cells.shape[0] == 0 or donor_cells.shape[0] == 0:
            continue
        else:
            rec_coords = np.stack((receiver_cells["x"].values, receiver_cells["y"].values))
            don_coords = np.stack((donor_cells["x"].values, donor_cells["y"].values))
            don_tree = cKDTree(don_coords.T)
            distances, indices = don_tree.query(rec_coords.T, k=1)
            cells_in = np.where(distances<dist_thresh)
            tot_in = cells_in[0].shape[0]
            cells_out = np.where(distances>dist_thresh)
            tot_out = cells_out[0].shape[0]

        if tot_in==0 or tot_out==0:
            continue

        rec_near_don = receiver_cells.iloc[cells_in]
        rec_near_don_pos = rec_near_don[rec_near_don[reciever_gene]>0]
        rec_near_don_neg = rec_near_don[rec_near_don[reciever_gene]==0]
        rec_coords_pos = np.stack((rec_near_don_pos["x"].values, rec_near_don_pos["y"].values, rec_near_don_pos[reciever_gene].values)).T
        rec_coords_neg = np.stack((rec_near_don_neg["x"].values, rec_near_don_neg["y"].values, rec_near_don_neg[reciever_gene].values)).T

        ########################################################################
        receiver_vals_pos = []
        donor_vals_pos = []
        for rec_coord in rec_coords_pos:
            x_rec = rec_coord[0]
            y_rec = rec_coord[1]
            within_thresh = donor_cells[
                np.sqrt((donor_cells['x'] - x_rec)**2 + (donor_cells['y'] - y_rec)**2) <= dist_thresh
            ]
    
            receiver_vals_pos.append(float(rec_coord[2]))
            donor_vals_pos.append(within_thresh.iloc[:,gene_start_index:].values.sum(axis=0))
        donor_vals_pos = np.asarray(donor_vals_pos).T
        receiver_vals_pos = np.asarray(receiver_vals_pos)
        ########################################################################
    
        ########################################################################
        receiver_vals_neg = []
        donor_vals_neg = []
        for rec_coord in rec_coords_neg:
            x_rec = rec_coord[0]
            y_rec = rec_coord[1]
            within_thresh = donor_cells[
                np.sqrt((donor_cells['x'] - x_rec)**2 + (donor_cells['y'] - y_rec)**2) <= dist_thresh
            ]
    
            receiver_vals_neg.append(float(rec_coord[2]))
            donor_vals_neg.append(within_thresh.iloc[:,gene_start_index:].values.sum(axis=0))
        donor_vals_neg = np.asarray(donor_vals_neg).T
        receiver_vals_neg = np.asarray(receiver_vals_neg)
        ########################################################################
    
        # donor_vals_neg = donor_vals_neg>0
        # donor_vals_pos = donor_vals_pos>0

        if donor_vals_pos.shape[0]!=5083 or donor_vals_neg.shape[0]!=5083:
            continue
            
        donor_values_pos[section_id] = donor_vals_pos
        donor_values_neg[section_id] = donor_vals_neg

    break

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21/21 [02:07<00:00,  6.09s/it]


In [39]:
array_list = [donor_values_pos[key] for key in donor_values_pos if key in colon_ids]
positive_cells = np.concatenate(array_list, axis=1).T

array_list = [donor_values_neg[key] for key in donor_values_neg if key in colon_ids]
negative_cells = np.concatenate(array_list, axis=1).T

neg_percents = 100*(negative_cells>0).sum(axis=0)/negative_cells.shape[0] # percentage of negative cells expression each gene
pos_percents = 100*(positive_cells>0).sum(axis=0)/positive_cells.shape[0] # percentage of positive cells expression each gene

fold_change_threshold = 1.5

for i in range(len(gene_names)):
    neg_percent = neg_percents[i]
    pos_percent = pos_percents[i]
    
    if not (pos_percent<60 and neg_percent<60): # one of the percentages is over 60%, so lets analyze
        if np.max((pos_percent/neg_percent, neg_percent/pos_percent))>fold_change_threshold: # there is a 1.5 fold change in percentage
            print(reciever_gene, gene_names[i], round(pos_percent, 2), round(neg_percent,2))

        ### else its not a simple percentage difference. Lets check for expression differences #####
        pos_mean_expr = positive_cells[:, i][positive_cells[:, i] > 0].mean()
        neg_mean_expr = negative_cells[:, i][negative_cells[:, i] > 0].mean()
        
        if pos_percent > 60 and neg_percent > 60:
            if np.max((neg_mean_expr / pos_mean_expr, pos_mean_expr / neg_mean_expr)) > fold_change_threshold:
                print(reciever_gene, gene_names[i], round(pos_percent, 2), round(neg_percent, 2),
                      round(pos_mean_expr, 2), round(neg_mean_expr, 2))
        
        if pos_percent > 60 and neg_percent < 60:
            if (pos_mean_expr / neg_mean_expr) > fold_change_threshold:
                print(reciever_gene, gene_names[i], round(pos_percent, 2), round(neg_percent, 2),
                      round(pos_mean_expr, 2), round(neg_mean_expr, 2))
        
        if pos_percent < 60 and neg_percent > 60:
            if (neg_mean_expr / pos_mean_expr) > fold_change_threshold:
                print(reciever_gene, gene_names[i], round(pos_percent, 2), round(neg_percent, 2),
                      round(pos_mean_expr, 2), round(neg_mean_expr, 2))


C7 NQO1 40.0 61.31
C7 TXNIP 61.11 38.76


In [40]:
consistently_expressed_gene_indices = []

for i in range(len(gene_names)):
    neg_percent = neg_percents[i]
    pos_percent = pos_percents[i]

    if pos_percent>60 or neg_percent>60:
        consistently_expressed_gene_indices.append(i)

consistently_expressed_genes = gene_names[consistently_expressed_gene_indices]
print(len(consistently_expressed_gene_indices))

229


In [38]:
# --- Config ---
use_parallel = False  # 🔁 Toggle this to enable/disable parallelization
n_jobs = 6          # Number of CPU cores to use (if parallel)
batch_size = 50000   # Adjust based on memory

pos_threshed = positive_cells[:, consistently_expressed_gene_indices] > 0
neg_threshed = negative_cells[:, consistently_expressed_gene_indices] > 0
gene_names_subset = np.array(gene_names)[consistently_expressed_gene_indices]

gate_funcs = {
    "AND":  lambda a,b: a & b,
    "OR":   lambda a,b: a | b,
    "XOR":  lambda a,b: a ^ b,
    "NAND": lambda a,b: ~(a & b),
    "NOR":  lambda a,b: ~(a | b),
    "XNOR": lambda a,b: ~(a ^ b),
}

threshold = 1.5
min_fraction = 0.6
num_genes = pos_threshed.shape[1]

pairs = np.array(list(itertools.combinations(range(num_genes), 2)))
total_pairs = pairs.shape[0]


def process_batch(batch_pairs):
    results = []
    idx1, idx2 = batch_pairs[:,0], batch_pairs[:,1]

    pos_1, pos_2 = pos_threshed[:, idx1], pos_threshed[:, idx2]
    neg_1, neg_2 = neg_threshed[:, idx1], neg_threshed[:, idx2]

    for gate_name, gate_func in gate_funcs.items():
        pos_gate = gate_func(pos_1, pos_2).mean(axis=0)
        neg_gate = gate_func(neg_1, neg_2).mean(axis=0)

        valid_mask = (pos_gate >= min_fraction) | (neg_gate >= min_fraction)
        nonzero_mask = neg_gate > 0
        ratio = np.zeros_like(pos_gate)
        ratio[nonzero_mask] = pos_gate[nonzero_mask] / neg_gate[nonzero_mask]

        hits_mask = valid_mask & ((ratio > threshold) | (neg_gate == 0))
        hits_idx = np.where(hits_mask)[0]

        for hit in hits_idx:
            i, j = idx1[hit], idx2[hit]
            pos_val, neg_val = pos_gate[hit], neg_gate[hit]
            ratio_val = ratio[hit] if neg_gate[hit] > 0 else np.inf
            results.append((gate_name, gene_names_subset[i], gene_names_subset[j], pos_val, neg_val, ratio_val))

    return results

# --- Process batches ---
batches = [pairs[start:start+batch_size] for start in range(0, total_pairs, batch_size)]

if use_parallel:
    print(f"Running with parallelization on {n_jobs} cores...")
    results = Parallel(n_jobs=n_jobs)(
        delayed(process_batch)(batch_pairs) for batch_pairs in tqdm(batches, desc="Parallel batches")
    )
else:
    print("Running without parallelization...")
    results = []
    for batch_pairs in tqdm(batches, desc="Serial batches"):
        results.append(process_batch(batch_pairs))

# --- Output ---
for batch_result in results:
    for gate_name, gene1, gene2, pos_val, neg_val, ratio_val in batch_result:
        print(f"{gate_name}: {gene1}, {gene2} | pos: {100*pos_val:.1f}, neg: {100*neg_val:.1f}, ratio: {ratio_val:.2f}")


Running without parallelization...


Serial batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.97s/it]

AND: ATIC, HSPD1 | pos: 63.3, neg: 42.0, ratio: 1.51
AND: ATIC, JUN | pos: 60.0, neg: 38.2, ratio: 1.57
AND: EEF1G, TXNIP | pos: 60.0, neg: 38.7, ratio: 1.55
AND: TGFBI, TXNIP | pos: 60.0, neg: 38.1, ratio: 1.58
NAND: ATP1A1, NQO1 | pos: 62.2, neg: 40.8, ratio: 1.52
NAND: CALM3, TSPAN8 | pos: 60.0, neg: 39.2, ratio: 1.53
NAND: CTTN, NBL1 | pos: 61.1, neg: 40.2, ratio: 1.52
NAND: CTTN, SQSTM1 | pos: 63.3, neg: 41.9, ratio: 1.51
NAND: CTTN, TSPAN8 | pos: 61.1, neg: 37.7, ratio: 1.62
NAND: CYCS, NBL1 | pos: 60.0, neg: 37.8, ratio: 1.59
NAND: DAP3, TSPAN8 | pos: 63.3, neg: 40.4, ratio: 1.57
NAND: EBP, TSPAN8 | pos: 60.0, neg: 39.3, ratio: 1.53
NAND: EEF1G, NQO1 | pos: 60.0, neg: 38.7, ratio: 1.55
NAND: EIF3C, NQO1 | pos: 61.1, neg: 40.6, ratio: 1.50
NAND: EMP2, TSPAN8 | pos: 68.9, neg: 45.5, ratio: 1.51
NAND: EPCAM, NQO1 | pos: 62.2, neg: 40.2, ratio: 1.55
NAND: HNRNPA1L2, NQO1 | pos: 62.2, neg: 40.6, ratio: 1.53
NAND: HSPA8, NQO1 | pos: 62.2, neg: 40.5, ratio: 1.54
NAND: LDHA, NQO1 | pos:

In [15]:
# --- Toggle for parallelization ---
threshold = 1.5  # fold-change threshold
min_fraction = 0.6  # require at least 60% coverage in pos or neg

use_parallel = False  # Set to False to disable parallel execution
batch_size = 50000
n_jobs = 32

# Create boolean (thresholded) versions
pos_threshed = positive_cells[:, consistently_expressed_gene_indices] > 0
neg_threshed = negative_cells[:, consistently_expressed_gene_indices] > 0

# Subset gene names accordingly
gene_names_subset = np.array(gene_names)[consistently_expressed_gene_indices]

# Define 3-argument logical gate functions
gates_3 = {
    "AND3":  lambda a, b, c: a & (b & c),
    "OR3":   lambda a, b, c: a | (b | c),
    "XOR3":  lambda a, b, c: np.logical_xor(a, np.logical_xor(b, c)),
    "NAND3": lambda a, b, c: ~(a & (b & c)),
    "NOR3":  lambda a, b, c: ~(a | (b | c)),
    "XNOR3": lambda a, b, c: ~(np.logical_xor(a, np.logical_xor(b, c))),
    # "MAJ": true if >=2 out of 3 are True
    "MAJ":   lambda a, b, c: ((a.astype(int) + b.astype(int) + c.astype(int)) >= 2),
}




# Prepare all triplets of gene indices (0..num_genes-1 in the subset)
num_genes_subset = pos_threshed.shape[1]
triplets = np.array(list(itertools.combinations(range(num_genes_subset), 3)))
total_triplets = triplets.shape[0]

def process_triplet_batch(batch_triplets):
    """
    For each triplet in 'batch_triplets', compute the specified 3-gene gates
    in positive/negative sets. Return all hits that pass filters.
    """
    results = []

    # Extract columns (pos_1, pos_2, pos_3) for the entire batch
    idx1 = batch_triplets[:, 0]
    idx2 = batch_triplets[:, 1]
    idx3 = batch_triplets[:, 2]

    # shape: (n_pos_cells, batch_size)
    pos_1 = pos_threshed[:, idx1]
    pos_2 = pos_threshed[:, idx2]
    pos_3 = pos_threshed[:, idx3]

    # shape: (n_neg_cells, batch_size)
    neg_1 = neg_threshed[:, idx1]
    neg_2 = neg_threshed[:, idx2]
    neg_3 = neg_threshed[:, idx3]

    # For each gate, compute fraction of cells that satisfy the condition
    for gate_name, gate_func in gates_3.items():
        # pos_gate[i] = fraction of pos cells for triplet i
        # Similarly for neg_gate
        pos_gate = gate_func(pos_1, pos_2, pos_3).mean(axis=0)
        neg_gate = gate_func(neg_1, neg_2, neg_3).mean(axis=0)

        # Filter out triplets that have < min_fraction in both pos & neg
        valid_mask = (pos_gate >= min_fraction) | (neg_gate >= min_fraction)

        # Evaluate fold-change (pos_gate / neg_gate)
        ratio = np.full_like(pos_gate, fill_value=0.0)
        nonzero_mask = (neg_gate > 0)
        ratio[nonzero_mask] = pos_gate[nonzero_mask] / neg_gate[nonzero_mask]

        # We treat "neg_gate == 0" separately => ratio = infinity if pos_gate > 0
        hits_mask = valid_mask & ((ratio > threshold) | (neg_gate == 0))

        hits_idx = np.where(hits_mask)[0]  # indices within this batch
        for hi in hits_idx:
            i1, i2, i3 = idx1[hi], idx2[hi], idx3[hi]
            p_val = pos_gate[hi]
            n_val = neg_gate[hi]
            ratio_val = ratio[hi] if n_val > 0 else float("inf")

            # Build a result record
            results.append((gate_name, i1, i2, i3, p_val, n_val, ratio_val))

    return results

total_triplets = triplets.shape[0]

# --- Split triplets into batches ---
batches = [triplets[start:start+batch_size] for start in range(0, total_triplets, batch_size)]


if use_parallel:
    print(f"Running with parallelization on {n_jobs} cores...")
    all_results = Parallel(n_jobs=n_jobs)(
        delayed(process_triplet_batch)(batch) for batch in tqdm(batches, desc="Parallel triplet batches")
    )
else:
    print("Running without parallelization...")
    all_results = []
    for batch in tqdm(batches, desc="Serial triplet batches"):
        all_results.append(process_triplet_batch(batch))

# Flatten results
all_results_flat = []
for batch_res in all_results:
    all_results_flat.extend(batch_res)

# Finally, print or handle them
for gate_name, i1, i2, i3, pos_val, neg_val, ratio_val in all_results_flat:
    print(
        f"{gate_name}: {gene_names_subset[i1]}, {gene_names_subset[i2]}, {gene_names_subset[i3]} "
        f"| pos: {pos_val:.3f}, neg: {neg_val:.3f}, ratio={ratio_val:.2f}"
    )


Running without parallelization...


Serial triplet batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:06<00:00,  6.79s/it]


In [ ]:
# ## check for logical gate interaction between pairs of genes. Are there two donor genes that interact to induce receiver gene?

# # Create boolean (thresholded) versions of the cell matrices:
# pos_threshed = positive_cells > 0
# neg_threshed = negative_cells > 0

# # Define a dictionary of logical gate functions.
# gates = {
#     "AND":  lambda a, b: np.logical_and(a, b),
#     "OR":   lambda a, b: np.logical_or(a, b),
#     "XOR":  lambda a, b: np.logical_xor(a, b),
#     "NAND": lambda a, b: np.logical_not(np.logical_and(a, b)),
#     "NOR":  lambda a, b: np.logical_not(np.logical_or(a, b)),
#     "XNOR": lambda a, b: np.logical_not(np.logical_xor(a, b))
# }

# threshold = 1.5  # Only report if fold-change exceeds 1.5

# # Iterate over every unique pair of genes in your consistently expressed set.
# total_combinations = comb(len(consistently_expressed_gene_indices), 2)
# for i, j in tqdm.tqdm(itertools.combinations(consistently_expressed_gene_indices, 2), total=total_combinations):
#     # For each logical gate, compute the fraction (mean) of cells that satisfy the condition.
#     for gate_name, gate_func in gates.items():
#         pos_val = gate_func(pos_threshed[:, i], pos_threshed[:, j]).mean()
#         neg_val = gate_func(neg_threshed[:, i], neg_threshed[:, j]).mean()

#         if pos_val<0.6 and neg_val<0.6: # one of them has to explain at least %60 of the cells
#             continue
            
#         # Avoid division by zero; if no negative cells, then its interesting anyway.
#         if neg_val == 0:
#             print(
#                 f"{gate_name}: {gene_names[i]}, "
#                 f"{gene_names[j]} | pos: {pos_val:.3f}, "
#                 f"neg: {neg_val:.3f}, ratio: {ratio:.2f}"
#             )
#             continue

#         ratio = pos_val / neg_val

#         if ratio > threshold:
#             print(
#                 f"{gate_name}: {gene_names[i]}, "
#                 f"{gene_names[j]} | pos: {pos_val:.3f}, "
#                 f"neg: {neg_val:.3f}, ratio: {ratio:.2f}"
#             )


In [ ]:
# from math import comb

# # Create boolean (thresholded) versions of the cell matrices:
# pos_threshed = (positive_cells > 0)
# neg_threshed = (negative_cells > 0)

# # Define 3-argument logical gate functions.
# gates_3 = {
#     "AND3":  lambda a, b, c: np.logical_and(a, np.logical_and(b, c)),
#     "OR3":   lambda a, b, c: np.logical_or(a, np.logical_or(b, c)),
#     "XOR3":  lambda a, b, c: np.logical_xor(a, np.logical_xor(b, c)),
#     "NAND3": lambda a, b, c: np.logical_not(np.logical_and(a, np.logical_and(b, c))),
#     "NOR3":  lambda a, b, c: np.logical_not(np.logical_or(a, np.logical_or(b, c))),
#     "XNOR3": lambda a, b, c: np.logical_not(np.logical_xor(a, np.logical_xor(b, c))),
#     # Example "majority" gate: True if at least 2/3 are True
#     "MAJ":   lambda a, b, c: ((a.astype(int) + b.astype(int) + c.astype(int)) >= 2)
# }

# threshold = 1.5  # Only report if fold-change exceeds 1.5

# # Iterate over every unique triplet of genes in consistently expressed set.
# total_combinations = comb(len(consistently_expressed_gene_indices), 3)
# for i, j, k in tqdm.tqdm(itertools.combinations(consistently_expressed_gene_indices, 3), total=total_combinations):
#     # For each gate, compute the fraction (mean) of cells that satisfy the condition.
#     for gate_name, gate_func in gates_3.items():
#         pos_val = gate_func(
#             pos_threshed[:, i],
#             pos_threshed[:, j],
#             pos_threshed[:, k]
#         ).mean()

#         neg_val = gate_func(
#             neg_threshed[:, i],
#             neg_threshed[:, j],
#             neg_threshed[:, k]
#         ).mean()

#         # Optional check: e.g., require pos_val or neg_val > 0.6, etc. 
#         if pos_val < 0.6 and neg_val < 0.6:
#             continue

#         # If no negative cells satisfy, could handle separately:
#         if neg_val == 0:
#             # If pos_val is nonzero, you might treat ratio as 'infinite'.
#             if pos_val > 0:
#                 print(
#                     f"{gate_name}: {gene_names[i]}, {gene_names[j]}, {gene_names[k]} "
#                     f"| pos: {pos_val:.3f}, neg: {neg_val:.3f}, ratio=∞"
#                 )
#             continue
        
#         ratio = pos_val / neg_val
        
#         if ratio > threshold:
#             print(
#                 f"{gate_name}: {gene_names[i]}, {gene_names[j]}, {gene_names[k]} "
#                 f"| pos: {pos_val:.3f}, neg: {neg_val:.3f}, ratio={ratio:.2f}"
#             )
